# DATA MINING EN ECONOMIA Y FINANZAS 2026
  Comisión jueves

# Arbol Impresion



### Seteo del ambiente en Google Colab

Esta parte se debe correr con el runtime en Python3
<br>Ir al menu, Runtime -> Change Runtime Type -> Runtime type ->  **Python 3**

Conectar la virtual machine donde está corriendo Google Colab con el  Google Drive, para poder tener persistencia de archivos.

In [1]:
# primero establecer el Runtime de Python 3
from google.colab import drive
drive.mount('/content/.drive')

Mounted at /content/.drive


los siguientes comandos estan en shell script de Linux

* Crear las carpetas en el Google Drive
* Bajar el competencia_01_crudo al Google Drive y tambien al disco local de la virtual machine que está corriendo Google Colab



In [2]:
%%shell

mkdir -p "/content/.drive/My Drive/dmeyf"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/dmeyf" /content/buckets/b1


mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets


# defino funcion descargar()
descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/dmeyf2026-9c6f/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}


# hago la descarga efectiva, llamando a descargar()
descargar  "competencia_01_crudo.csv"

## Generacion de la clase_ternaria

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

In [10]:
getwd()
list.files()
file.exists("gridsearch_detalle.txt")

[1] "/content"

[1] "buckets"     "datasets"    "sample_data"

[1] FALSE

In [22]:
require( "data.table" )

# leo el dataset
dataset <- fread("/content/datasets/competencia_01_crudo.csv" )

# calculo el periodo0 consecutivo
dsimple <- dataset[, list(
    "pos" = .I,
    numero_de_cliente,
    periodo0 = as.integer(foto_mes/100)*12 +  foto_mes%%100 ) ]


# ordeno
setorder( dsimple, numero_de_cliente, periodo0 )

# calculo topes
periodo_ultimo <- dsimple[, max(periodo0) ]
periodo_anteultimo <- periodo_ultimo - 1


# calculo los leads de orden 1 y 2
dsimple[, c("periodo1", "periodo2") :=
    shift(periodo0, n=1:2, fill=NA, type="lead"),  numero_de_cliente ]

# assign most common class values = "CONTINUA"
dsimple[ periodo0 < periodo_anteultimo, clase_ternaria := "CONTINUA" ]

# calculo BAJA+1
dsimple[ periodo0 < periodo_ultimo &
    ( is.na(periodo1) | periodo0 + 1 < periodo1 ),
    clase_ternaria := "BAJA+1" ]

# calculo BAJA+2
dsimple[ periodo0 < periodo_anteultimo & (periodo0+1 == periodo1 )
    & ( is.na(periodo2) | periodo0 + 2 < periodo2 ),
    clase_ternaria := "BAJA+2" ]


# pego el resultado en el dataset original y grabo
setorder( dsimple, pos )
dataset[, clase_ternaria := dsimple$clase_ternaria ]

fwrite( dataset,
    file =  "/content/datasets/competencia_01.csv.gz",
    sep = ","
)

## Impresion del arbol

limpio el ambiente de R

In [23]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,748046,40.0,1473300,78.7,1473300,78.7
Vcells,1407197,10.8,171050480,1305.1,213764585,1630.9


In [24]:
# cargo las librerias que necesito
require("data.table")
require("rpart")
require("parallel")

if(!require("rpart.plot")) install.packages("rpart.plot")
require("rpart.plot")

if(!require("R.utils")) install.packages("R.utils")
require("R.utils")


Loading required package: rpart

Loading required package: parallel

Loading required package: rpart.plot

Warning message in library(package, lib.loc = lib.loc, character.only = TRUE, logical.return = TRUE, :
“there is no package called ‘rpart.plot’”
Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Loading required package: rpart.plot

Loading required package: R.utils

Warning message in library(package, lib.loc = lib.loc, character.only = TRUE, logical.return = TRUE, :
“there is no package called ‘R.utils’”
Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘R.oo’, ‘R.methodsS3’


Loading required package: R.utils

Loading required package: R.oo

Loading required package: R.methodsS3

R.methodsS3 v1.8.2 (2022-06-13 22:00:14 UTC) successfully loaded. See ?R.methodsS3 for help.

R.oo v1.27.1 (2025-05-02 21:00:05 UTC) successfully loaded. See ?R.oo for help.


Attaching package: ‘R.oo’




In [25]:
PARAM <- list()
PARAM$experimento <- "arbol0301"


PARAM$param_basicos <- list(
  "cp"= -1,
  "maxdepth"= 4,
  "minsplit"= 20,
  "minbucket"= 5
)



In [ ]:
# Carpeta del experimento
setwd("/content/buckets/b1/exp")
dir.create(PARAM$experimento, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", PARAM$experimento ))

In [26]:
# lectura del dataset
# dataset <- fread("/content/datasets/competencia_01.csv.gz")
require( "data.table" )
dataset1 <- fread("./buckets/b1/exp/HT2900/gridsearch_detalle.txt" )
dataset2 <- fread("./buckets/b1/exp/HT2900/gridsearch_detalle_1sem.txt" )
dataset3 <- fread("./buckets/b1/exp/HT2901/gridsearch_detalle.txt" )
dataset <- rbind(dataset1, dataset2, dataset3)



In [28]:
dataset

semilla,cp,maxdepth,minsplit,minbucket,ganancia_test
<int>,<dbl>,<int>,<int>,<int>,<dbl>
389713,-2.5,3,1750,875,386925000
620377,-2.5,3,1750,875,393341667
736889,-2.5,3,1750,875,342191667
389713,-2.0,3,1750,875,386925000
620377,-2.0,3,1750,875,393341667
736889,-2.0,3,1750,875,342191667
389713,-1.5,3,1750,875,386925000
620377,-1.5,3,1750,875,393341667
736889,-1.5,3,1750,875,342191667


In [29]:
  modelo <- rpart("ganancia_test ~ .",
    data = dataset,
    xval = 0,
    control = PARAM$param_basicos
  ) #

In [30]:
  # impresion del arbol en un pdf
  arch_arbol <- "arbol.pdf"
  pdf(file = arch_arbol, width=28, height=4)
  prp(modelo, extra=101, digits=5, branch=1, type=4, varlen=0, faclen=0)
  dev.off()

agg_record_125f633708fd 
                      2